<a href="https://colab.research.google.com/github/demerchantsean-wq/Exercise-Dashboard/blob/main/Exercise_Dashboard.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. Install the required packages (runs silently)
!pip install -q gpxpy osmnx PyWavelets scipy

import gpxpy
import pandas as pd
import numpy as np
import folium
import branca.colormap as cm
from folium.features import ColorLine
from google.colab import drive
import os
import re
import json
from datetime import datetime
import ipywidgets as widgets
from IPython.display import display, clear_output
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.colors as mcolors
import osmnx as ox
import pywt
from scipy.signal import correlate, correlation_lags
import requests

# Vectorized Haversine formula to calculate distance between GPS points
def calculate_haversine_distance(lon1, lat1, lon2, lat2):
    lon1, lat1, lon2, lat2 = map(np.radians, [lon1, lat1, lon2, lat2])
    dlon = lon2 - lon1
    dlat = lat2 - lat1
    a = np.sin(dlat/2.0)**2 + np.cos(lat1) * np.cos(lat2) * np.sin(dlon/2.0)**2
    c = 2 * np.arcsin(np.sqrt(a))
    return 3956 * c

# Signal Denoising using PyWavelets
def denoise_signal(series, wavelet='sym4'):
    data = series.fillna(0).values
    if len(data) == 0: return series
    coeffs = pywt.wavedec(data, wavelet, mode='per')
    sigma = np.median(np.abs(coeffs[-1])) / 0.6745
    if sigma == 0: sigma = 1e-9
    threshold = sigma * np.sqrt(2 * np.log(len(data)))
    coeffs[1:] = [pywt.threshold(c, value=threshold, mode='soft') for c in coeffs[1:]]
    return pd.Series(pywt.waverec(coeffs, wavelet, mode='per')[:len(data)])

# 2. Mount Drive
print("Connecting to Google Drive...")
drive.mount('/content/drive')

# --- CONFIGURATION VARIABLES ---
BASE_FEATURES = [
    'hr', 'elevation_ft', 'slope_smoothed', 'elapsed_minutes',
    'mph_smoothed', 'hr_denoised', 'hr_aligned', 'slope_denoised'
]

DEFAULT_SELECTED_FEATURES = [
    'hr', 'elevation_ft', 'slope_smoothed', 'elapsed_minutes',
    'mph_smoothed', 'hr_aligned', 'slope_denoised'
]

# 3. Build the Folder Browser UI
start_path = '/content/drive/MyDrive/Biometrics and Environmental Data/EKG/Fourth Frontier'
if not os.path.exists(start_path):
    start_path = '/content'

path_label = widgets.Text(value=start_path, description='Current Dir:', disabled=True, layout=widgets.Layout(width='80%'))
dir_select = widgets.Select(options=[], description='Folders:', layout=widgets.Layout(width='80%', height='150px'))
btn_up = widgets.Button(description='⬆️ Up Level', button_style='warning')
btn_refresh = widgets.Button(description='🔄 Refresh Folder', button_style='success')

gpx_dropdown = widgets.Dropdown(options=[], description='GPX File:', disabled=True, layout=widgets.Layout(width='80%'))
hr_dropdown = widgets.Dropdown(options=['None'], description='Heart Rate:', disabled=True, layout=widgets.Layout(width='80%'))
rr_dropdown = widgets.Dropdown(options=['None'], description='R-R Interval:', disabled=True, layout=widgets.Layout(width='80%'))
summary_dropdown = widgets.Dropdown(options=['None'], description='Summary:', disabled=True, layout=widgets.Layout(width='80%'))

btn_load = widgets.Button(description='📂 Load Data & Extract Features', button_style='primary', disabled=True)
feature_box = widgets.VBox([])
btn_plot = widgets.Button(description='🗺️ Generate Maps & Interactive Plot', button_style='info', disabled=True)
out = widgets.Output()

# Globals for passing data between Load and Plot steps
global_df = pd.DataFrame()
global_route_points = []
global_bounds = []
global_summary_cols = []
checkbox_dict = {}
default_features_list = []

feature_labels = {
    'hr': 'Heart Rate',
    'elevation_ft': 'Elevation',
    'slope_smoothed': 'Slope (%)',
    'elapsed_minutes': 'Elapsed Time',
    'mph_smoothed': 'Velocity (MPH)',
    'hr_denoised': 'Denoised Heart Rate',
    'hr_aligned': 'Aligned Heart Rate',
    'slope_denoised': 'Denoised Slope (%)'
}

def on_scan_clicked(b=None):
    with out:
        clear_output()
        folder = path_label.value
        try:
            gpx_files = [f for f in os.listdir(folder) if f.lower().endswith('.gpx')]
            csv_files = [f for f in os.listdir(folder) if f.lower().endswith('.csv')]
            if gpx_files:
                gpx_dropdown.options = sorted(gpx_files)
                gpx_dropdown.disabled = False
                btn_load.disabled = False
            else:
                gpx_dropdown.options = []
                gpx_dropdown.disabled = True
                btn_load.disabled = True

            csv_options = ['None'] + sorted(csv_files)
            is_csv_empty = len(csv_files) == 0

            hr_dropdown.options = csv_options
            hr_dropdown.disabled = is_csv_empty
            if '1sec_hr.csv' in csv_files: hr_dropdown.value = '1sec_hr.csv'

            rr_dropdown.options = csv_options
            rr_dropdown.disabled = is_csv_empty
            if 'rr_interval.csv' in csv_files: rr_dropdown.value = 'rr_interval.csv'

            summary_dropdown.options = csv_options
            summary_dropdown.disabled = is_csv_empty
            if 'summarydata.csv' in csv_files: summary_dropdown.value = 'summarydata.csv'

            print(f"Found {len(gpx_files)} GPX files and {len(csv_files)} CSV files.")
            print("Assign your files below and click 'Load Data & Extract Features'.")
            feature_box.children = []
            btn_plot.disabled = True
        except Exception as e:
            print("Could not read directory contents.")

def on_dir_change(change):
    if change['new']:
        new_path = os.path.join(path_label.value, change['new'])
        update_browser(new_path)

def update_browser(path):
    try:
        path_label.value = path
        items = sorted([d for d in os.listdir(path) if os.path.isdir(os.path.join(path, d)) and not d.startswith('.')])
        dir_select.unobserve(on_dir_change, names='value')
        dir_select.options = items
        dir_select.value = None
        dir_select.observe(on_dir_change, names='value')
        on_scan_clicked()
    except Exception as e:
        dir_select.unobserve(on_dir_change, names='value')
        dir_select.options = []
        dir_select.value = None
        dir_select.observe(on_dir_change, names='value')

def on_up_clicked(b):
    new_path = os.path.dirname(path_label.value)
    update_browser(new_path)

def load_csv(dropdown, name, folder):
    if dropdown.value != 'None':
        path = os.path.join(folder, dropdown.value)
        print(f"Loading {name} data from '{dropdown.value}'...")
        try:
            return pd.read_csv(path)
        except Exception as e:
            print(f"❌ Error loading {name} CSV: {e}")
    return None

def process_data(b):
    global global_df, global_route_points, global_bounds, global_summary_cols
    global checkbox_dict, default_features_list

    with out:
        clear_output()
        folder = path_label.value

        hr_df = load_csv(hr_dropdown, "Heart Rate", folder)
        rr_df = load_csv(rr_dropdown, "R-R Interval", folder)
        summary_df = load_csv(summary_dropdown, "Summary Data", folder)

        gpx_path = os.path.join(folder, gpx_dropdown.value)
        print(f"Processing GPX route '{gpx_dropdown.value}'...\n")
        with open(gpx_path, 'r') as gpx_file:
            gpx = gpxpy.parse(gpx_file)

        route_data = []
        for track in gpx.tracks:
            for segment in track.segments:
                for point in segment.points:
                    route_data.append({
                        'lat': point.latitude,
                        'lon': point.longitude,
                        'elevation': point.elevation,
                        'time': point.time
                    })
        df = pd.DataFrame(route_data)
        if df.empty:
            print("Error: No track points found in this GPX file.")
            return

        df['elevation'] = df['elevation'].interpolate().bfill()
        df['elevation_ft'] = df['elevation'] * 3.28084
        summary_cols_added = []

        if not df['time'].isnull().all():
            df['time'] = pd.to_datetime(df['time'], utc=True)
            df = df.sort_values('time')

            if hr_df is not None and not hr_df.empty:
                if any('session_' in col for col in hr_df.columns):
                    def get_session_num(col_name):
                        match = re.search(r'session_(\d+)', col_name)
                        return int(match.group(1)) if match else -1
                    session_cols = sorted([c for c in hr_df.columns if 'session_' in c], key=get_session_num)
                    hr_values = hr_df[session_cols].melt()['value'].dropna().astype(float).values
                    gpx_start = df['time'].iloc[0]
                    hr_times = gpx_start + pd.to_timedelta(np.arange(len(hr_values)), unit='s')
                    hr_processed = pd.DataFrame({'merge_time': hr_times, 'hr': hr_values})
                else:
                    time_col = hr_df.columns[0]
                    hr_col = hr_df.columns[1]
                    hr_processed = pd.DataFrame()
                    hr_processed['merge_time'] = pd.to_datetime(hr_df[time_col], utc=True, errors='coerce')
                    hr_processed['hr'] = hr_df[hr_col]

                hr_processed = hr_processed.dropna(subset=['merge_time']).sort_values('merge_time')
                df = pd.merge_asof(
                    df, hr_processed[['merge_time', 'hr']],
                    left_on='time', right_on='merge_time', direction='nearest', tolerance=pd.Timedelta('15s')
                )
                df['hr'] = df['hr'].fillna(0)
            else:
                df['hr'] = 0

            if summary_df is not None and not summary_df.empty:
                time_col_sum = next((col for col in summary_df.columns if 'time' in col.lower()), summary_df.columns[0])
                summary_df['merge_time_sum'] = pd.to_datetime(summary_df[time_col_sum], utc=True, errors='coerce')
                summary_df = summary_df.dropna(subset=['merge_time_sum']).sort_values('merge_time_sum')
                all_num_cols = summary_df.select_dtypes(include=[np.number]).columns.tolist()
                num_cols = [col for col in all_num_cols if col != time_col_sum]
                rename_dict = {col: f"Summary_{col}" for col in num_cols}
                summary_df_clean = summary_df[['merge_time_sum'] + num_cols].rename(columns=rename_dict)

                df = pd.merge_asof(
                    df, summary_df_clean,
                    left_on='time', right_on='merge_time_sum', direction='nearest', tolerance=pd.Timedelta('15s')
                )
                summary_cols_added = list(rename_dict.values())
                df[summary_cols_added] = df[summary_cols_added].fillna(0)

                for p_col in [c for c in summary_cols_added if 'power' in c.lower()]:
                    df[p_col + '_cumsum'] = df[p_col].cumsum()

            df['time_pacific'] = df['time'].dt.tz_convert('US/Pacific')
            df['time_str'] = df['time_pacific'].dt.strftime('%I:%M:%S %p %Z')
            df['elapsed_minutes'] = (df['time'] - df['time'].iloc[0]).dt.total_seconds() / 60.0
            df['elapsed_minutes'] = df['elapsed_minutes'].interpolate().bfill()

            df['dist_miles'] = calculate_haversine_distance(
                df['lon'].shift(1), df['lat'].shift(1), df['lon'], df['lat']
            )
            df['time_diff_hours'] = df['time'].diff().dt.total_seconds() / 3600.0
            df['mph'] = df['dist_miles'] / df['time_diff_hours']
            df['mph'] = df['mph'].replace([np.inf, -np.inf], np.nan).fillna(0)
            df['mph_smoothed'] = df['mph'].rolling(window=10, min_periods=1).mean()

            df['elev_diff_ft'] = df['elevation_ft'].diff()
            df['dist_ft'] = df['dist_miles'] * 5280.0
            df['slope_pct'] = (df['elev_diff_ft'] / df['dist_ft']) * 100
            df['slope_pct'] = df['slope_pct'].replace([np.inf, -np.inf], np.nan).fillna(0)
            df['slope_smoothed'] = df['slope_pct'].rolling(window=10, min_periods=1).mean()

            df['slope_denoised'] = denoise_signal(df['slope_pct'])
            df['hr_denoised'] = denoise_signal(df['hr'])

            norm_slope = df['slope_denoised'] - df['slope_denoised'].mean()
            norm_hr = df['hr_denoised'] - df['hr_denoised'].mean()

            if norm_hr.any() and norm_slope.any():
                corr = correlate(norm_hr, norm_slope, mode='full')
                lags = correlation_lags(len(norm_hr), len(norm_slope), mode='full')

                valid_indices = np.where((lags >= 0) & (lags <= 60))[0]
                if len(valid_indices) > 0:
                    optimal_lag = lags[valid_indices[np.argmax(corr[valid_indices])]]
                else:
                    optimal_lag = 0
            else:
                optimal_lag = 0

            df['hr_aligned'] = df['hr'].shift(-optimal_lag).ffill().bfill().fillna(0)

            hr_bins = [0, 119, 136, 153, 171, 300]
            df['metabolic_zone'] = pd.cut(df['hr_aligned'], bins=hr_bins, labels=['Recovery', 'Fat Oxidation', 'Mixed Fuel', 'Lactate Threshold', 'Anaerobic'])

        else:
            for col in ['elapsed_minutes', 'mph_smoothed', 'slope_smoothed', 'hr', 'hr_aligned', 'slope_denoised', 'hr_denoised']: df[col] = 0
            df['time_str'] = "N/A"
            df['metabolic_zone'] = 'Recovery'

        global_df = df
        global_route_points = list(zip(df['lat'], df['lon']))
        global_bounds = [[df['lat'].min(), df['lon'].min()], [df['lat'].max(), df['lon'].max()]]
        global_summary_cols = summary_cols_added

        available_features = BASE_FEATURES + summary_cols_added
        default_summary_cols = [c for c in summary_cols_added if 'qos' not in c.lower()]
        default_features_list = DEFAULT_SELECTED_FEATURES + default_summary_cols

        checkbox_dict.clear()

        def create_feature_rows(feat_list):
            rows = [widgets.HBox([
                widgets.Label('Feature', layout=widgets.Layout(width='220px', font_weight='bold')),
                widgets.Label('Map', layout=widgets.Layout(width='60px', font_weight='bold')),
                widgets.Label('Plot', layout=widgets.Layout(width='60px', font_weight='bold'))
            ])]
            for f in feat_list:
                clean_name = feature_labels.get(f, f.replace("Summary_", ""))
                is_def = f in default_features_list
                cb_m = widgets.Checkbox(value=is_def, indent=False, layout=widgets.Layout(width='60px'))
                cb_p = widgets.Checkbox(value=is_def, indent=False, layout=widgets.Layout(width='60px'))
                checkbox_dict[f] = {'map': cb_m, 'plot': cb_p}
                rows.append(widgets.HBox([widgets.Label(clean_name, layout=widgets.Layout(width='220px')), cb_m, cb_p]))
            return widgets.VBox(rows, layout=widgets.Layout(margin='0 40px 0 0'))

        half = len(available_features) // 2 + len(available_features) % 2
        left_box = create_feature_rows(available_features[:half])
        right_box = create_feature_rows(available_features[half:])
        ui_grid = widgets.HBox([left_box, right_box], layout=widgets.Layout(width='100%', justify_content='flex-start'))

        btn_all = widgets.Button(description="✅ Select All", button_style='success')
        btn_none = widgets.Button(description="❌ Select None", button_style='danger')
        btn_def = widgets.Button(description="🔄 Default", button_style='info')

        def set_all(b):
            for cbs in checkbox_dict.values():
                cbs['map'].value = True; cbs['plot'].value = True

        def set_none(b):
            for cbs in checkbox_dict.values():
                cbs['map'].value = False; cbs['plot'].value = False

        def set_default(b):
            for f, cbs in checkbox_dict.items():
                is_def = f in default_features_list
                cbs['map'].value = is_def; cbs['plot'].value = is_def

        btn_all.on_click(set_all)
        btn_none.on_click(set_none)
        btn_def.on_click(set_default)

        feature_box.children = [
            widgets.HTML("<h3>Select Features to Map & Plot:</h3>"),
            widgets.HBox([btn_all, btn_none, btn_def]),
            ui_grid
        ]

        btn_plot.disabled = False
        print("✅ Data successfully loaded and features extracted.")

def generate_visualizations(b):
    with out:
        clear_output()
        df = global_df
        folder = path_label.value

        map_feats = [f for f, cbs in checkbox_dict.items() if cbs['map'].value]
        plot_feats = [f for f, cbs in checkbox_dict.items() if cbs['plot'].value]

        if not map_feats and not plot_feats:
            print("❌ No features selected. Please select at least one Map or Plot before generating.")
            return

        cyclosm_tiles = 'https://{s}.tile-cyclosm.openstreetmap.fr/cyclosm/{z}/{x}/{y}.png'
        cyclosm_attr = 'CyclOSM | © OpenStreetMap contributors'
        timestamp_str = datetime.now().strftime('%Y%m%d%H%M%S')

        all_selected_feats = list(set(map_feats + plot_feats))
        colormaps = {}
        for feat in all_selected_feats:
            if feat == 'hr':
                if not df['hr'].eq(0).all():
                    valid_hr = df[df['hr'] > 0]['hr'].tolist()
                    min_hr, max_hr = min(valid_hr), max(valid_hr)
                    mid_green = 142.5
                    if max_hr <= mid_green: c_colors, c_index = ['yellow', 'green'], [min_hr, max_hr]
                    elif min_hr >= mid_green: c_colors, c_index = ['green', 'red'], [min_hr, max_hr]
                    else: c_colors, c_index = ['yellow', 'green', 'red'], [min_hr, mid_green, max_hr]
                    cmap = cm.LinearColormap(colors=c_colors, index=c_index, vmin=min_hr, vmax=max_hr)
                    cmap.caption = "hr - Heart Rate"
                    colormaps['hr'] = cmap
                continue

            if feat in df.columns and not df[feat].isnull().all() and (df[feat] != 0).any():
                metrics = df[feat].tolist()
                min_m, max_m = min(metrics), max(metrics)
                if min_m == max_m: max_m = min_m + 1.0

                if 'slope' in feat.lower():
                    max_abs = max(abs(min_m), abs(max_m))
                    if max_abs == 0: max_abs = 1.0
                    cmap = cm.linear.viridis.scale(-max_abs, max_abs)
                else:
                    cmap = cm.linear.viridis.scale(min_m, max_m)

                clean_name = feature_labels.get(feat, feat.replace("Summary_", ""))
                cmap.caption = f"{feat} - {clean_name}"
                colormaps[feat] = cmap

        # --- PRE-CALCULATE TOOLTIP HTML FOR MAP HOVERS ---
        tooltip_array = []
        for _, row in df.iterrows():
            html = f"<div style='font-size: 12px; line-height: 1.4;'><b>Clock:</b> {row['time_str']}<br>"
            for f_tip in all_selected_feats:
                val = row[f_tip]
                c_name = feature_labels.get(f_tip, f_tip.replace("Summary_", ""))
                if f_tip == 'hr':
                    c_color = colormaps['hr'](val) if 'hr' in colormaps and val > 0 else '#808080'
                    hr_disp = f"{val:.0f} bpm" if val > 0 else "N/A"
                    html += f"<span style='color:{c_color}; font-size:14px;'>■</span> <b>HR:</b> {hr_disp} ({row.get('metabolic_zone', 'N/A')})<br>"
                elif 'power' in f_tip.lower():
                    c_color = colormaps[f_tip](val) if f_tip in colormaps else '#000'
                    html += f"<span style='color:{c_color}; font-size:14px;'>■</span> <b>{c_name}:</b> {val:.2f} (Total: {row.get(f_tip + '_cumsum', 0):.2f})<br>"
                else:
                    c_color = colormaps[f_tip](val) if f_tip in colormaps else '#000'
                    html += f"<span style='color:{c_color}; font-size:14px;'>■</span> <b>{c_name}:</b> {val:.1f}<br>"
            html += "</div>"
            tooltip_array.append(html)
        df['tooltip_html'] = tooltip_array

        ts_feats = [f for f in plot_feats if f in df.columns and f != 'elapsed_minutes']
        has_plots = len(ts_feats) > 0

        if has_plots:
            req_cols = list(set(['lat', 'lon', 'elapsed_minutes', 'tooltip_html'] + ts_feats))
            js_data = df[req_cols].to_json(orient='records')
            labels_json = json.dumps([feature_labels.get(f, f.replace("Summary_", "")) for f in ts_feats])
            feats_json = json.dumps(ts_feats)
            plot_height = 400

        # --- PAIRGRID SCATTER & CUSTOM HISTOGRAMS ---
        if plot_feats:
            print("\n--- GENERATING TIME-CODED SCATTER & HISTOGRAM GRID ---")
            scatter_cols = [f for f in plot_feats if 'cumsum' not in f.lower() and f != 'elapsed_minutes']
            valid_plot_cols = [c for c in scatter_cols if c in df.columns and not df[c].isnull().all() and (df[c] != 0).any()]

            if not valid_plot_cols:
                print("Not enough varied data to generate scatter plots.")
            else:
                time_col = 'elapsed_minutes'
                df_plot = df.dropna(subset=valid_plot_cols + [time_col]).reset_index(drop=True)
                np.random.seed(42)
                df_jittered = df_plot.copy()
                for col in valid_plot_cols:
                    col_range = df_plot[col].max() - df_plot[col].min()
                    if col_range == 0: col_range = 1
                    noise = np.random.uniform(-0.015 * col_range, 0.015 * col_range, size=len(df_plot))
                    df_jittered[col] = df_plot[col] + noise

                scatter_cmap = plt.get_cmap('viridis')
                norm = plt.Normalize(df_jittered[time_col].min(), df_jittered[time_col].max())
                colors = scatter_cmap(norm(df_jittered[time_col]))
                num_time_bins = 15
                df_jittered['time_bin'] = pd.cut(df_jittered[time_col], bins=num_time_bins, labels=False)
                df_jittered['time_bin'] = df_jittered['time_bin'].fillna(0).astype(int)
                bin_centers = []
                for i in range(num_time_bins):
                    bin_data = df_jittered[df_jittered['time_bin'] == i][time_col]
                    if not bin_data.empty: bin_centers.append(bin_data.mean())
                    else: bin_centers.append(0)
                bin_colors = [scatter_cmap(norm(c)) for c in bin_centers]

                plt.figure(figsize=(16, 16))
                g = sns.PairGrid(df_jittered, vars=valid_plot_cols)

                def hollow_scatter(x, y, **kwargs):
                    c = colors[x.index]
                    plt.scatter(x, y, facecolors='none', edgecolors=c, s=20, alpha=0.6, linewidths=1.0)
                g.map_offdiag(hollow_scatter)

                def time_coded_4_way(x, **kwargs):
                    ax = plt.gca()
                    for spine in ['top', 'right', 'bottom', 'left']: ax.spines[spine].set_visible(False)
                    ax.set_yticks([])
                    ax.patch.set_alpha(0.0)
                    ax_bl = ax.inset_axes([0.0, 0.0, 0.48, 0.48])
                    ax_tl = ax.inset_axes([0.0, 0.52, 0.48, 0.48])
                    ax_br = ax.inset_axes([0.52, 0.0, 0.48, 0.48])
                    ax_tr = ax.inset_axes([0.52, 0.52, 0.48, 0.48])
                    for sub_ax in [ax_bl, ax_tl, ax_br, ax_tr]:
                        sub_ax.set_xticks([]); sub_ax.set_yticks([])
                    bins = np.histogram_bin_edges(x.dropna(), bins=20)
                    bins_data = []
                    for i in range(num_time_bins):
                        data_slice = x[df_jittered['time_bin'] == i].dropna()
                        if not data_slice.empty: bins_data.append(data_slice)
                        else: bins_data.append(pd.Series([], dtype=float))
                    ax_bl.hist(x.dropna(), bins=bins, color='slategray', edgecolor='none', alpha=0.9)
                    for i in range(num_time_bins):
                        if len(bins_data[i]) > 0: ax_tl.hist(bins_data[i], bins=bins, color=bin_colors[i], alpha=0.5, edgecolor='none')
                    for i in reversed(range(num_time_bins)):
                        if len(bins_data[i]) > 0: ax_br.hist(bins_data[i], bins=bins, color=bin_colors[i], alpha=0.5, edgecolor='none')
                    counts_list = []
                    for data_slice in bins_data:
                        counts, _ = np.histogram(data_slice, bins=bins)
                        counts_list.append(counts)
                    counts_matrix = np.array(counts_list)
                    total_counts = counts_matrix.sum(axis=0)
                    with np.errstate(divide='ignore', invalid='ignore'):
                        fractions_matrix = np.true_divide(counts_matrix, total_counts)
                        fractions_matrix[~np.isfinite(fractions_matrix)] = 0
                    bottom = np.zeros(len(bins)-1)
                    widths = np.diff(bins)
                    centers = bins[:-1] + widths/2
                    for i in range(num_time_bins):
                        ax_tr.bar(centers, fractions_matrix[i], width=widths, bottom=bottom, color=bin_colors[i], edgecolor='none', alpha=0.9, align='center')
                        bottom += fractions_matrix[i]
                    ax_tr.set_ylim(0, 1.05)
                g.map_diag(time_coded_4_way)

                g.fig.subplots_adjust(right=0.91)
                cbar_ax = g.fig.add_axes([0.93, 0.15, 0.02, 0.7])
                sm = plt.cm.ScalarMappable(cmap=scatter_cmap, norm=norm)
                sm.set_array([])
                cbar = g.fig.colorbar(sm, cax=cbar_ax)
                cbar.set_label('Elapsed Time (minutes)', rotation=270, labelpad=25)

                save_filename = f"ExerciseDashboardHistogram_{timestamp_str}.png"
                save_path = os.path.join(folder, save_filename)
                g.savefig(save_path, dpi=150, bbox_inches='tight')
                print(f"✅ Successfully saved histogram to: {save_path}")
                plt.show()

            # --- EFFORT TOPOGRAPHY VISUALIZATIONS ---
            req_features = ['slope_denoised', 'mph_smoothed', 'hr_aligned']
            if all(f in plot_feats for f in req_features):
                print("\n--- GENERATING EFFORT TOPOGRAPHY VISUALIZATIONS ---")
                fig, ax = plt.subplots(1, 2, figsize=(18, 6))

                hb = ax[0].hexbin(df['slope_denoised'], df['mph_smoothed'], C=df['hr_aligned'],
                                  gridsize=25, cmap='inferno', reduce_C_function=np.mean)
                ax[0].set_title('Effort Topography: Gradient vs. Velocity')
                ax[0].set_xlabel('Denoised Gradient (%)')
                ax[0].set_ylabel('Velocity (MPH)')
                fig.colorbar(hb, ax=ax[0], label='Mean Aligned Heart Rate (bpm)')

                sns.kdeplot(data=df, x='slope_denoised', y='mph_smoothed',
                            fill=True, cmap='mako', ax=ax[1], thresh=0.05)
                ax[1].set_title('Density Histogram: Gradient vs. Velocity')
                ax[1].set_xlabel('Denoised Gradient (%)')

                plt.tight_layout()
                plt.show()

        # --- INDIVIDUAL LINKED MAP/PLOT GENERATION LOOP ---
        map_counter = 1
        for feat in all_selected_feats:
            show_map = checkbox_dict[feat]['map'].value

            if not show_map: continue

            clean_name = feature_labels.get(feat, feat.replace("Summary_", ""))
            print(f"\n--- MAP {map_counter}: {clean_name.upper()} ---")

            fig_height = 400
            if has_plots: fig_height += plot_height + 20

            f_feat = folium.Figure(width='75%', height=fig_height)
            m = folium.Map(tiles=cyclosm_tiles, attr=cyclosm_attr, width='100%', height=400)

            if feat == 'hr':
                if 'hr' not in colormaps:
                    folium.PolyLine(global_route_points, color='red', weight=6, opacity=0.8).add_to(m)
                else:
                    ColorLine(positions=global_route_points, colors=df['hr'].tolist(), colormap=colormaps['hr'], weight=6).add_to(m)
                    colormaps['hr'].add_to(m)

                north, south = df['lat'].max() + 0.005, df['lat'].min() - 0.005
                east, west = df['lon'].max() + 0.005, df['lon'].min() - 0.005
                overpass_url = "https://overpass-api.de/api/interpreter"
                headers = {'User-Agent': 'CyclingPerformanceDashboard/1.0 (Colab)'}
                overpass_query = f"""
                [out:json][timeout:25];
                (
                  node["amenity"="drinking_water"]({south},{west},{north},{east});
                  way["shop"="bicycle"]({south},{west},{north},{east});
                  node["shop"="bicycle"]({south},{west},{north},{east});
                );
                out center;
                """
                try:
                    response = requests.post(overpass_url, data={'data': overpass_query}, headers=headers, timeout=30)
                    if response.status_code == 200:
                        data = response.json()
                        for element in data.get('elements', []):
                            if element['type'] == 'node': lat, lon = element['lat'], element['lon']
                            elif 'center' in element: lat, lon = element['center']['lat'], element['center']['lon']
                            else: continue

                            tags = element.get('tags', {})
                            is_water = tags.get('amenity') == 'drinking_water'
                            icon_color, icon_type = ('blue', 'tint') if is_water else ('orange', 'bicycle')
                            name = tags.get('name', 'Drinking Water' if is_water else 'Bike Shop')

                            folium.Marker(location=[lat, lon], tooltip=name, icon=folium.Icon(color=icon_color, icon=icon_type, prefix='fa')).add_to(m)
                    else:
                        print(f"Overpass POI retrieval failed with HTTP {response.status_code}")
                except Exception as e:
                    print(f"Overpass POI retrieval failed: {e}")
            else:
                ColorLine(positions=global_route_points, colors=df[feat].tolist(), colormap=colormaps[feat], weight=6).add_to(m)
                colormaps[feat].add_to(m)

            folium.Marker(global_route_points[0], tooltip="Start", icon=folium.Icon(color='green', icon='play')).add_to(m)
            folium.Marker(global_route_points[-1], tooltip="End", icon=folium.Icon(color='red', icon='stop')).add_to(m)

            m.fit_bounds(global_bounds)
            m.add_to(f_feat)
            map_id = m.get_name()
            m.save(os.path.join(folder, f"Map_{feat}_{timestamp_str}.html"))

            # --- INJECT JAVASCRIPT FOR MULTI-AXIS HOVER SYNC PLOT ---
            plot_div_id = f"plot_{map_counter}_{timestamp_str}"

            custom_html = f"""<script src="https://cdn.plot.ly/plotly-2.24.1.min.js"></script>"""

            if has_plots:
                custom_html += f"""<div id="{plot_div_id}" style="width: 100%; height: {plot_height}px; margin-bottom: 10px; background: white;"></div>"""

            custom_html += f"""
            <script>
            (function() {{
                var telemetryData = {js_data};
                var myPlot = document.getElementById('{plot_div_id}');
            """

            if has_plots:
                custom_html += f"""
                document.body.insertBefore(myPlot, document.body.firstChild);
                var features = {feats_json};
                var featureLabels = {labels_json};
                var colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728', '#9467bd', '#8c564b', '#e377c2', '#7f7f7f', '#bcbd22', '#17becf'];

                var traces = [];
                var numFeats = features.length;
                var leftAxes = Math.ceil(numFeats / 2);
                var rightAxes = Math.floor(numFeats / 2);
                var offsetStep = 0.06;
                var domainStart = (leftAxes > 1) ? (leftAxes - 1) * offsetStep : 0;
                var domainEnd = (rightAxes > 1) ? 1.0 - (rightAxes - 1) * offsetStep : 1.0;

                var layout = {{
                    title: 'Interactive Telemetry: Hover over the Map or Plot to Synchronize',
                    // Force hovermode to 'x' to show individual tags for every plot line
                    hovermode: 'x',
                    xaxis: {{
                        title: 'Elapsed Time (minutes)',
                        domain: [domainStart, domainEnd],
                        showgrid: true,
                        // Enable the black dashed crosshair spikeline
                        showspikes: true,
                        spikemode: 'across',
                        spikedash: 'dash',
                        spikecolor: 'black',
                        spikethickness: 2
                    }},
                    margin: {{l: 60, r: 60, t: 40, b: 40}}
                }};

                for (var i=0; i<numFeats; i++) {{
                    var isLeft = (i % 2 === 0);
                    var axisIndex = Math.floor(i / 2);
                    var yAxisName = i === 0 ? 'y' : 'y' + (i + 1);

                    traces.push({{
                        x: telemetryData.map(d => d.elapsed_minutes),
                        y: telemetryData.map(d => d[features[i]]),
                        mode: 'lines',
                        name: featureLabels[i],
                        line: {{color: colors[i % colors.length], width: 1.5}},
                        yaxis: yAxisName
                    }});

                    var axisObj = {{
                        title: {{text: featureLabels[i], font: {{color: colors[i % colors.length], size: 10}}}},
                        tickfont: {{color: colors[i % colors.length], size: 9}},
                        showgrid: (i === 0),
                        zeroline: (i === 0),
                        side: isLeft ? 'left' : 'right'
                    }};

                    if (i !== 0) {{ axisObj.overlaying = 'y'; }}

                    if (axisIndex > 0) {{
                        axisObj.anchor = 'free';
                        axisObj.position = isLeft ? (domainStart - axisIndex * offsetStep) : (domainEnd + axisIndex * offsetStep);
                    }} else {{
                        axisObj.anchor = 'x';
                    }}

                    if (i === 0) {{ layout.yaxis = axisObj; }}
                    else {{ layout['yaxis' + (i + 1)] = axisObj; }}
                }}

                Plotly.newPlot(myPlot, traces, layout);
                """

            custom_html += f"""
                var isSyncing = false;

                var initMapSync = setInterval(function() {{
                    var myMap = window['{map_id}'];
                    if (myMap || '{map_id}' === 'null') {{
                        clearInterval(initMapSync);

                        var hoverMarker = null;
                        if (myMap) {{
                            hoverMarker = L.circleMarker([0, 0], {{
                                color: 'black', fillColor: 'white', fillOpacity: 1, radius: 7, weight: 2, interactive: false
                            }}).addTo(myMap);

                            var tooltip = L.tooltip({{direction: 'right', offset: [10, 0], opacity: 0.95}});
                            hoverMarker.bindTooltip(tooltip);

                            // Map -> Plotly listener
                            myMap.getContainer().addEventListener('mousemove', function(e) {{
                                if (isSyncing) return;
                                var rect = myMap.getContainer().getBoundingClientRect();
                                var x = e.clientX - rect.left;
                                var y = e.clientY - rect.top;
                                var latlng = myMap.containerPointToLatLng([x, y]);

                                var lat = latlng.lat;
                                var lon = latlng.lng;
                                var minDist = Infinity;
                                var minIdx = -1;
                                var cosLat = Math.cos(lat * Math.PI / 180.0);

                                for(var i=0; i<telemetryData.length; i++) {{
                                    var dlat = telemetryData[i].lat - lat;
                                    var dlon = (telemetryData[i].lon - lon) * cosLat;
                                    var dist = dlat*dlat + dlon*dlon;
                                    if(dist < minDist) {{
                                        minDist = dist;
                                        minIdx = i;
                                    }}
                                }}

                                if (minDist < 0.00005) {{
                                    isSyncing = true;

                                    hoverMarker.setLatLng([telemetryData[minIdx].lat, telemetryData[minIdx].lon]);
                                    hoverMarker.setTooltipContent(telemetryData[minIdx].tooltip_html);
                                    if (!hoverMarker.isTooltipOpen()) hoverMarker.openTooltip();

                                    if (myPlot) {{
                                        // Triggering by xval directly invokes the axis spikeline logic and hits all curves natively
                                        Plotly.Fx.hover(myPlot, {{ xval: telemetryData[minIdx].elapsed_minutes }});
                                    }}

                                    isSyncing = false;
                                }} else {{
                                    hoverMarker.closeTooltip();
                                    if (myPlot) Plotly.Fx.unhover(myPlot);
                                }}
                            }});

                            myMap.getContainer().addEventListener('mouseleave', function(e) {{
                                hoverMarker.closeTooltip();
                                if (myPlot) Plotly.Fx.unhover(myPlot);
                            }});
                        }}

                        // Plotly -> Map listener
                        if (myPlot) {{
                            myPlot.on('plotly_hover', function(data){{
                                if (isSyncing) return;
                                isSyncing = true;
                                var pt = data.points[0].pointIndex;
                                if (myMap) {{
                                    hoverMarker.setLatLng([telemetryData[pt].lat, telemetryData[pt].lon]);
                                    hoverMarker.setTooltipContent(telemetryData[pt].tooltip_html);
                                    if (!hoverMarker.isTooltipOpen()) hoverMarker.openTooltip();
                                }}
                                isSyncing = false;
                            }});

                            myPlot.on('plotly_unhover', function(data){{
                                if (isSyncing) return;
                                if (myMap) hoverMarker.closeTooltip();
                            }});
                        }}
                    }}
                }}, 200);
            }})();
            </script>
            """

            f_feat.get_root().html.add_child(folium.Element(custom_html))
            display(f_feat)
            map_counter += 1

# 4. Wire up buttons and initialize
btn_up.on_click(on_up_clicked)
btn_refresh.on_click(on_scan_clicked)
btn_load.on_click(process_data)
btn_plot.on_click(generate_visualizations)
dir_select.observe(on_dir_change, names='value')

# Display layout
display(path_label)
display(widgets.HBox([dir_select, widgets.VBox([btn_up, btn_refresh])]))
display(widgets.VBox([gpx_dropdown, hr_dropdown, rr_dropdown, summary_dropdown]))
display(btn_load)
display(feature_box)
display(btn_plot)
display(out)

# Initialize the first view and trigger the auto-scan
update_browser(start_path)

Connecting to Google Drive...
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Text(value='/content/drive/MyDrive/Biometrics and Environmental Data/EKG/Fourth Frontier', description='Curren…

Button(button_style='primary', description='📂 Load Data & Extract Features', disabled=True, style=ButtonStyle(…

VBox()

Button(button_style='info', description='🗺️ Generate Maps & Interactive Plot', disabled=True, style=ButtonStyl…

Output()